# Day 5：亲眼看懂 MLA 的 latent 与矩阵吸收

配套 [直观讲义](../docs/MLA_INTUITION.md)。MLA = Multi-head Latent Attention。
每层每个历史 token 都有自己的短向量 c；不同头用不同投影生成 K/V 内容，位置 Key 另存。
本实验使用同一组随机低秩参数，比较展开路径与紧凑路径。CPU 即可，不下载模型。
它不是任意普通 Attention 的无损压缩，也不验证 DeepSeek 的质量或加速比。

In [ ]:
import os
import sys
from pathlib import Path

root = Path.cwd()
if root.name == 'learning':
    root = root.parent
if not (root / 'learning/mla_algebra.py').is_file():
    raise RuntimeError('Open from repository root or learning directory.')
os.chdir(root)
if str(root) not in sys.path:
    sys.path.insert(0, str(root))

from learning.mla_algebra import compare_paths

## 1. 两个数怎样产生不同头的 K/V？

假设 c=(a,b)=(2,3)，头0的 K=(a,b,a+b)，V=(2a,3b)；头1的 K=(a+b,a,b)，V=(3a,2b)。
两头展开共有10个数，但它们只由两个数决定。生成规则是共享模型权重，不是每 token 缓存。
对于头0和当前 q=(1,2,1)，先预测 q·K，与 (2,3)·c 是否一样。这里暂不考虑 RoPE。

In [ ]:
a, b = 2, 3
k0, v0 = (a, b, a + b), (2 * a, 3 * b)
k1, v1 = (a + b, a, b), (3 * a, 2 * b)
q = (1, 2, 1)
expanded_score = sum(qi * ki for qi, ki in zip(q, k0))
latent_score = 2 * a + 3 * b
print('c:', (a, b))
print('head0:', k0, v0, 'head1:', k1, v1)
print('expanded dot:', expanded_score, 'latent dot:', latent_score)
assert expanded_score == latent_score == 13

## 2. 带独立 RoPE 分支的完整 Attention 小算例

固定7个历史位置、4个头、latent rank=3、每头内容K维=4、位置K维=2、V维=3。

- 路径A：展开历史内容K和V，拼接位置K，softmax 后加权V。
- 路径B：将当前Q变换到latent空间，内容分数+位置分数，softmax 后加权latent，再做V投影。

先预测：两条路径的分数、权重、输出是否一致？latent 是 `[7,3]`，不是 `[1,3]`。
注意：两条路径都保留所有历史位置，并执行同样的缩放和softmax。

In [ ]:
report = compare_paths()
for key in ('latent_shape', 'position_key_shape', 'expanded_key_shape',
            'expanded_value_shape', 'latent_query_shape', 'output_shape'):
    print(key, report[key])
print('score max error:', report['score_max_error'])
print('output max error:', report['output_max_error'])
print('head0 expanded:', report['expanded_head0_output'])
print('head0 latent:  ', report['latent_head0_output'])
assert report['score_alignment'] == report['output_alignment'] == 'passed'
assert report['latent_shape'] == [7, 3]

## 3. 缓存到底存几个数？

显式拼接每头完整K/V：每token `4×((4+2)+3)=36` 个数。
紧凑路径：每token `latent 3 + 位置K 2 = 5` 个数。
这里展开路径会把共享位置K重复到各头；如果位置K单独共享，展开内容K/V加位置K是 `4×(4+3)+2=30` 个数。
36对5只是本实验物化数组的比较，不是 DeepSeek 的真实压缩率。
实际推理分配还要考虑精度、padding、量化scale、分页等；下面不计参数、临时张量和allocator。

In [ ]:
print('expanded elements/token:', report['expanded_elements_per_token'])
print('compact elements/token: ', report['compact_elements_per_token'])
print('7 tokens, FP32 expanded bytes:', report['expanded_bytes'])
print('7 tokens, FP32 compact bytes: ', report['compact_bytes'])
assert report['expanded_bytes'] == 7 * 36 * 4 == 1008
assert report['compact_bytes'] == 7 * 5 * 4 == 140
longer = compare_paths(tokens=14)
assert longer['compact_bytes'] == 2 * report['compact_bytes']
assert longer['expanded_bytes'] == 2 * report['expanded_bytes']
print('T翻倍：两种历史状态都变两倍，MLA没有删除token轴。')

## 4. 一次只改一个维度

先预测：头数4变8，固定rank时紧凑状态每token会变吗？rank从3变6又如何？
每次调用重新生成参数，只对齐该次调用内部的两条路径。不能比较不同架构输出是否一样，也不能据此声称质量相同。

In [ ]:
for heads, rank in ((4, 3), (8, 3), (4, 6)):
    result = compare_paths(heads=heads, rank=rank)
    print('heads=', heads, 'rank=', rank,
          'expanded/token=', result['expanded_elements_per_token'],
          'compact/token=', result['compact_elements_per_token'],
          'alignment=', result['output_alignment'])
print(report['scope'])

## 自测与参考答案

**1. 7个token是否只有一个latent？** 不是，每层7个位置各一个，本例shape为`[7,3]`，Batch轴省略。

**2. 不同头的K/V是不是完全相同？** 不是，共享生成来源c，各头投影不同；GQA才是分组直接共享K/V。

**3. 为什么不必每次展开全部历史内容K？** `qᵀ(Wc)=(Wᵀq)ᵀc`，先变换当前Q，再与历史c打分。

**4. V为何也能不全部展开？** `Σ a_t(Wv c_t)=Wv(Σ a_t c_t)`，线性投影可放在加权汇集之后。

**5. 缓存有没有历史Q？** 没有。本例紧凑状态是每位置latent加位置Key；当前Q用于查询历史。

**6. 这证明任意MHA都能无损改为MLA吗？** 不能。只证明同一套已定义为低秩的参数在两条路径下等价；浮点存在误差。

**7. 已证明生产加速或DeepSeek模型效果了吗？** 没有。此处没有训练、query压缩、Norm、最终out投影、完整Cache更新、生产kernel或真实checkpoint。

**8. heads=8/rank=3与heads=4/rank=6时如何记账？** 前者展开72/紧凑5；后者展开36/紧凑8个数每token，均为本例设定。

原理依据：DeepSeek-V2论文2.1节及附录C；DeepSeek-V3官方推理代码。地址与推导见配套讲义。当前个人进度以 [进度记录](../docs/LEARNING_PROGRESS.md) 为准。